Your current approach is called Stationary Noise Estimation, which assumes the "noise floor" (the baseline noise level) never changes after the first half-second . As you noticed with the coffee machine, this fails because the coffee machine is non-stationary—its statistical properties (volume and pitch) change over time .

To fix this, you need an Adaptive Noise Estimator. Instead of taking a single "snapshot" at the beginning, the algorithm must "track" the noise as the audio plays  

1. The Strategy: Minimum Statistics Tracking
The most common way to handle changing noise (like a coffee machine or traffic) is to assume that speech is intermittent, but noise is persistent.


The Logic: In any 1–2 second window of audio, there are tiny gaps between words or syllables .

The Method: The algorithm looks for the minimum energy in each frequency band over a sliding window. Since speech is loud and bursty, the "minimum" level found in that window is almost certainly just the background noise .

In [11]:
import numpy as np
import librosa
import soundfile as sf
from scipy.ndimage import median_filter, uniform_filter1d

def denoise_adaptive(filename, output_name):
    y, sr = librosa.load(filename, sr=None)
    stft_full = librosa.stft(y, n_fft=2048, hop_length=512)
    s_full, phase = librosa.magphase(stft_full)

    # 1. Switch to Power Domain for better noise separation
    power_spec = s_full**2

    # 2. BETTER NOISE ESTIMATION
    # Instead of minimum, use a median filter across time (axis 1)
    # This captures the 'average' background noise much better.
    window_size = int(librosa.time_to_frames(1.5, sr=sr, hop_length=512))
    
    # We estimate noise as the median energy in each frequency band over time
    # Using a 2D median filter or 1D median on the time axis
    noise_est = median_filter(power_spec, size=(1, window_size))

    # 3. Aggressive Subtraction
    # Increase alpha if noise persists (try 2.0 to 4.0)
    alpha = 3.0  
    beta = 0.05  # Lower floor for a "darker" background
    
    # Subtract noise in power domain
    subtracted = power_spec - (alpha * noise_est)
    
    # Apply the mask
    # We ensure we don't go below our 'beta' floor
    mask = np.maximum(subtracted, beta * power_spec) / (power_spec + 1e-10)
    mask = np.sqrt(np.clip(mask, 0, 1)) # Back to magnitude domain

    # 4. Smoothing (Crucial for "musical noise" reduction)
    mask = uniform_filter1d(mask, size=5, axis=1) # Time smoothing
    mask = uniform_filter1d(mask, size=3, axis=0) # Freq smoothing

    # 5. Reconstruct
    s_clean = s_full * mask
    y_clean = librosa.istft(s_clean * phase, hop_length=512)
    
    # Normalize to prevent clipping
    y_clean = librosa.util.normalize(y_clean)
    sf.write(output_name, y_clean, sr)

In [12]:
input_file = "./data/noise/audio_1.wav"
output = "test_adaptive.wav"
denoise_adaptive(input_file, output)

4. Why this works for the Coffee Machine
Since the coffee machine might get louder or quieter, the minimum_filter1d will see the "valleys" between your words rise and fall with the machine.

When the machine grinds louder, the "minimum" value in your 1.5-second window goes up.

The mask then automatically subtracts more during that period.

When the machine stops, the minimum drops, and the mask stops subtracting as much.